## Hands-On Lab 4 - Feature Engineering

In this lab, you will experiment with improving the lagged random forest model from Lab #3 by adding endogenous and exogenous features.

### Step 1 - Loading the Training Data

Run the following code to load the Bike Sharing training data and set the index.

In [ ]:
import pandas as pd

# Load training data
bike_sharing_train = pd.read_csv('BikeSharing_Train.csv', parse_dates = ['date_time'])

# Set index
bike_sharing_train = bike_sharing_train.set_index('date_time')
bike_sharing_train = bike_sharing_train.asfreq('h')
bike_sharing_train = bike_sharing_train.sort_index()

### Step 2 - Loading the Validation Data

Run the following code to load the Bike Sharing validation data and set the index.

In [ ]:
# Load validation data
bike_sharing_val = pd.read_csv('BikeSharing_Validation.csv', parse_dates = ['date_time'])

# Set index
bike_sharing_val = bike_sharing_val.set_index('date_time')
bike_sharing_val = bike_sharing_val.asfreq('h')
bike_sharing_val = bike_sharing_val.sort_index()

### Step 3 - Chart Set Up

Run the following code to set up line charts using dark mode.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from skforecast.plot import set_dark_theme

# Set fonts/grid defaults
custom_style = {
    'axes.titlesize': 18,
    'axes.labelsize': 15,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'grid.alpha': 0.3,
}

# Make dark mode the default for all plots
set_dark_theme(custom_style = custom_style) 

### Step 4 - Forecast KPI Calculations

Run the following code for the utility function that calculates forecasting model KPIs (i.e., bias and MAE).

In [ ]:
import numpy as np

# Utility function for our model KPIs
def forecast_kpis(forecasts, targets):
    # Bias calculations
    errors = forecasts - targets 
    targets_sum = np.sum(targets)
    bias = np.mean(errors)
    bias_pct = np.sum(errors) / targets_sum 

    # MAE calculations
    mae = np.mean(np.abs(errors))
    mae_pct = np.sum(np.abs(errors)) / targets_sum

    # Display
    print(f'Bias: {bias:0.4f}, {(bias_pct * 100):0.2f}%')
    print(f'MAE: {mae:0.4f}, {(mae_pct * 100):0.2f}%') 

### Step 5 - Lagged RF Forecasting Model

Run the following code to train the lagged random forest model from Lab #3. You will use this model's performance as a reference for feature engineering in this lab.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from skforecast.recursive import ForecasterRecursive

# Forecast horizon
h = 168

# Create a list of the "optimal" lags from PACF
pacf_lags = [1, 2, 5, 10, 15, 18, 22, 25]

# Forecast using a Random Forest "optimal" lags
rf_pacf_lags = ForecasterRecursive(regressor = RandomForestRegressor(random_state = 12345), lags = pacf_lags)

# Learn from the lagged dataset
rf_pacf_lags.fit(y = bike_sharing_train['users'])

# Make predictions to compare against validation set
rf_pacf_lags_preds = rf_pacf_lags.predict(steps = h)

# What is the accuracy on the validation set?
forecast_kpis(rf_pacf_lags_preds, bike_sharing_val['users'])

### Step 6 - Endogenous Features

As covered in the lecture, endogenous features are created from target values. The *RollingFeatures* class provides functionality for creating a variety of endogenous features. From the PACF analysis in Lab 3, you know that the first five lags are highly associated with the target. Therefore, it makes sense to experiment with some features within a five-lag window. Enter and run the following code.

In [ ]:
# Enter your lab code here

### Step 7 - Train RF Lagged + Rolling Feature Model

Enter and run the following code to train and evaluate a random forest forecasting model using the new rolling endogenous features.

In [ ]:
# Enter your lab code here

### Step 8 - Visualize the RF Lagged + Rolling Model

Run the following code to visualize the new random forest model's forecasts versus the validation dataset observations.

In [ ]:
# Create a plot 11 inches wide and 5 inches high
fig, ax = plt.subplots(figsize = (11, 5))

# Validation targets
bike_sharing_val['users'].head(n = h).plot(ax = ax, linewidth = 2)

# Forecasts
rf_lags_roll_preds.plot(ax = ax, linewidth = 2, linestyle = '--', 
                        label = 'RF Lagged + Roll Forecast')

# Display plot
ax.set_title('RF Lagged + Rolling Features Forecasts for Validation Dataset')
ax.legend()
fig.tight_layout()
plt.show()

### Step 9 - Exogenous Features

Given the nature of the Bike Sharing dataset, exogenous features are likely to improve the forecasting model. For example, the following features could be predictive of the number of users:

- *holiday* - Binary indicator.
- *workingday* - Binary indicator.

These features are included in the training, validation, and test datasets. They can be used as-is without any encoding. Enter and run the following code to train a model to use these three exogenous features. 

In [ ]:
# Enter your lab code here

### Step 10 - Seasonal Exogenous Features (Training)

The Bike Sharing dataset is at the hourly grain. However, given the nature of the business problem, it's likely that there is intraweek seasonality in the data. For example, fewer users on weekends. Technically, the *weekday* feature should be treated as a categorical feature and one-hot encoded. Enter and run the following code to create a new training DataFrame.

In [ ]:
# Enter your lab code here

### Step 11 - Seasonal Exogenous Features (Validation)

Any feature engineering performed on the training dataset must be mirrored in the validation dataset as well. Run the following code to create a new dataset for validation.

In [ ]:
# One-hot encode the features and create a new DataFrame for validation
bike_sharing_exog_val = pd.get_dummies(bike_sharing_val, columns = categorical_features, 
                                       prefix = categorical_features, dtype = int)
bike_sharing_exog_val.info()

### Step 12 - Seasonal Exogenous Features Revisited

With the addition of new exogenous features, another iteration of model training is required to assess any improvements. Enter and run the following code.

In [ ]:
# Enter your lab code here

### Step 13 - Visualize Revised Seasonal Exogenous Features

Run the following code to visualize the latest random forest model's forecasts versus the validation dataset observations.

In [ ]:
# Create a plot 11 inches wide and 5 inches high
fig, ax = plt.subplots(figsize = (11, 5))

# Validation targets
bike_sharing_val['users'].head(n = h).plot(ax = ax, linewidth = 2)

# Forecasts
rf_lags_roll_exog_preds.plot(ax = ax, linewidth = 2, linestyle = '--', 
                             label = 'RF Lagged + Roll + Exog Forecast')

# Display plot
ax.set_title('RF Lagged + Rolling + Exogenous Features Forecasts for Validation Dataset')
ax.legend()
fig.tight_layout()
plt.show()